# Blob to Bronze â€” Transactions Auto Loader

Reads CSV files arriving at `/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/` using Databricks Auto Loader and writes them to the Delta table `retail_q_dab_dev.blob_bronze.transactions`.

In [0]:
# â”€â”€ Auto Loader: CSV â†’ Delta bronze table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
source_path       = "/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/"
schema_location   = "/Volumes/retail_q_dab_dev/volumes/blob_source/_schemas/"
checkpoint_path   = "/Volumes/retail_q_dab_dev/volumes/blob_source/_checkpoints/"
target_table      = "retail_q_dab_dev.blob_bronze.transactions"

# Define schema explicitly â€” avoids a full data scan for inference in streaming
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

csv_schema = StructType([
    StructField("transaction_id",       StringType(),  True),
    StructField("opportunity_name",     StringType(),  True),
    StructField("product_id",            StringType(),  True),
    StructField("store_id",              StringType(),  True),
    StructField("quantity",              IntegerType(), True),
    StructField("selling_price",         IntegerType(), True),
    StructField("discount_amount",       IntegerType(), True),
    StructField("transaction_timestamp", StringType(),  True),
    StructField("payment_mode",          StringType(),  True),
    StructField("sales_channel",         StringType(),  True),
])

# Read CSV files incrementally via Auto Loader
raw_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.schemaHints", 
                "transaction_id string, opportunity_name string, "
                "product_id string, store_id string, "
                "quantity int, selling_price int, discount_amount int, "
                "transaction_timestamp string, payment_mode string, "
                "sales_channel string")
        .option("header", "true")
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace", "true")
        .option("mode", "PERMISSIVE")
        .schema(csv_schema)
        .load(source_path)
)

# Add bronze-layer metadata columns
from pyspark.sql.functions import current_timestamp, col

bronze_df = (
    raw_df
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
)

# Write to the Delta target table via foreachBatch (append mode)
(
    bronze_df.writeStream
        .foreachBatch(lambda batch_df, batch_id: (
            batch_df.write
                .mode("append")
                .option("mergeSchema", "true")
                .saveAsTable(target_table)
        ))
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
)

print("Auto Loader stream started. Use the next cell to verify loaded data.")

Auto Loader stream started. Use the next cell to verify loaded data.


In [0]:
# â”€â”€ Verify the data landed in the bronze table â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
spark.sql(f"SELECT COUNT(*) AS row_count FROM {target_table}").display()
spark.sql(f"SELECT * FROM {target_table} LIMIT 10").display()

row_count
1100


transaction_id,opportunity_name,product_id,store_id,quantity,selling_price,discount_amount,transaction_timestamp,payment_mode,sales_channel,_source_file,_ingestion_timestamp
TXN000001,Royal Lifestyle Order 84,P0011,S005,5,93681,9545,17-Mar-2026 05.09.51 PM,Card,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000002,Quick Mobile Order 47,P0020,S004,1,22723,2060,13-Jan-2026 01.57.14 PM,Cash,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000003,Fresh Appliances Order 15,null,S001,3,24335,3346,13-Jan-2026 04.48.13 PM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000004,Urban Furniture Order 35,P0015,S003,3,51821,5958,09-Mar-2026 12.38.06 AM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000005,Smart Furniture Order 63,P0005,S004,4,92199,12352,23-Jan-2026 09.35.51 AM,Card,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000006,Royal Electronics Order 11,P0003,S001,3,1409,135,09-Mar-2026 11.11.34 AM,Card,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000007,Value Mobile Order 81,P0008,S003,2,38121,1763,18-Feb-2026 12.46.09 AM,UPI,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000008,Value Fashion Order 95,P0006,S002,2,107459,14564,17-Jan-2026 06.12.09 PM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000009,Value Fashion Order 4,P0006,S005,1,51355,973,23-Feb-2026 03.50.51 PM,Cash,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000010,Quick Appliances Order 22,P0009,S004,2,75727,627,27-Jan-2026 10.00.57 PM,Net Banking,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z


In [0]:
spark.sql(f"SELECT * FROM {target_table}").display()

transaction_id,opportunity_name,product_id,store_id,quantity,selling_price,discount_amount,transaction_timestamp,payment_mode,sales_channel,_source_file,_ingestion_timestamp
TXN000001,Royal Lifestyle Order 84,P0011,S005,5,93681,9545,17-Mar-2026 05.09.51 PM,Card,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000002,Quick Mobile Order 47,P0020,S004,1,22723,2060,13-Jan-2026 01.57.14 PM,Cash,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000003,Fresh Appliances Order 15,null,S001,3,24335,3346,13-Jan-2026 04.48.13 PM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000004,Urban Furniture Order 35,P0015,S003,3,51821,5958,09-Mar-2026 12.38.06 AM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000005,Smart Furniture Order 63,P0005,S004,4,92199,12352,23-Jan-2026 09.35.51 AM,Card,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000006,Royal Electronics Order 11,P0003,S001,3,1409,135,09-Mar-2026 11.11.34 AM,Card,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000007,Value Mobile Order 81,P0008,S003,2,38121,1763,18-Feb-2026 12.46.09 AM,UPI,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000008,Value Fashion Order 95,P0006,S002,2,107459,14564,17-Jan-2026 06.12.09 PM,Net Banking,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000009,Value Fashion Order 4,P0006,S005,1,51355,973,23-Feb-2026 03.50.51 PM,Cash,Online,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
TXN000010,Quick Appliances Order 22,P0009,S004,2,75727,627,27-Jan-2026 10.00.57 PM,Net Banking,Store,/Volumes/retail_q_dab_dev/volumes/blob_source/transactions_source/05_blob_transactions_history.csv,2026-10-07T10:24:00.916Z
